# Notebook 4 — RAG Trust and Evaluation

This notebook validates the AI explanation layer separately from the forecasting engine. It measures structured routing, FAISS retrieval, numerical fidelity, scope control, and Gemini availability. It does not retrain the forecasting models and it does not establish clinical validation.

## Step 1 — Obtain a temporary Colab copy of the GitHub repository

Cloning downloads the public repository into the temporary Colab runtime. It does not give Colab permission to change GitHub.

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/NNS1619/oncology-demand-intelligence.git"
REPOSITORY_DIR = Path("/content/oncology-demand-intelligence")

if not REPOSITORY_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

os.chdir(REPOSITORY_DIR)
print("Working directory:", Path.cwd())
print("Repository files found:", len(list(Path.cwd().rglob("*"))))

## Step 2 — Install the repository requirements

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Requirements installed.")

## Step 3 — Load the Gemini key securely

In Colab, open the key icon in the left sidebar, create a secret named `GOOGLE_API_KEY`, paste the Gemini key there, and enable notebook access. The following cell copies it into runtime memory only. It never prints or writes the key.

In [ ]:
from google.colab import userdata

google_api_key = userdata.get("GOOGLE_API_KEY")
if not google_api_key:
    raise RuntimeError("Colab secret GOOGLE_API_KEY was not found.")

os.environ["GOOGLE_API_KEY"] = google_api_key
del google_api_key
print("Gemini key loaded into temporary runtime memory. The key was not displayed.")

## What the three modes test

- **Static:** no API calls; checks the evidence corpus and exact CSV router.
- **Retrieval:** uses embeddings and FAISS; checks whether the expected evidence source appears in the top five chunks.
- **Full:** uses the router, embeddings, FAISS, and Gemini; checks answer content, saved-number fidelity, refusal behavior, and availability.

The holdout split is used for the final API-dependent test so the answers were not used to design the router.

In [ ]:
subprocess.run(
    [sys.executable, "rag/evaluate_rag.py", "--mode", "static"],
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "rag/evaluate_rag.py",
        "--mode", "retrieval",
        "--split", "holdout",
    ],
    check=True,
)

## Step 4 — Run the full holdout evaluation

The seven-second pause reduces rate-limit risk on free or low-throughput Gemini tiers. This final run overwrites the earlier summary with the complete holdout result.

In [ ]:
subprocess.run(
    [
        sys.executable, "rag/evaluate_rag.py",
        "--mode", "full",
        "--split", "holdout",
        "--delay-seconds", "7",
    ],
    check=True,
)

## Step 5 — Review results before publishing them

A `REVIEW` status is useful information, not something to hide. Inspect failed questions and improve the corpus, router, or prompt for a documented reason. Do not tune against the holdout answers repeatedly; move newly discovered cases into development and create a fresh holdout set.

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv("data/outputs/rag_evaluation_summary.csv")
details = pd.read_csv("data/outputs/rag_evaluation_details.csv")

display(summary)

review_rows = details.loc[
    (details["response_success"].fillna(1) < 1)
    | (details["source_recall_at_k"].fillna(1) < 1)
    | (details["numerical_exactness"].fillna(1) < 1)
    | (details["refusal_pass"].fillna(1) < 1)
]
print("Questions requiring review:", len(review_rows))
display(review_rows[["test_id", "question", "run_status", "actual_sources", "answer"]])

## Step 6 — Download the two evaluated outputs

Upload these two generated CSVs to `data/outputs/` in GitHub and commit them. Streamlit will then display the latest saved evaluation in its AI Trust panel.

In [ ]:
from google.colab import files

files.download("data/outputs/rag_evaluation_summary.csv")
files.download("data/outputs/rag_evaluation_details.csv")